# Smart Greenhouse Persistence, GRU and LSTM Baselines

## 00 - Experiment Overview

Notebook nay load lai locked split va train-fitted scalers cua notebook 01, sau do huan luyen mot GRU
va mot LSTM dung chung protocol. Persistence la scientific reference. Model selection chi dung validation
2024; Test A/B/C duoc mo sau khi best checkpoints da freeze.

> Local validator chi chay CPU smoke. Full training duoc thuc hien tren Google Colab GPU.

## 01 - Environment & Training Configuration

Tat ca training hyperparameters va smoke limits nam tai mot cell. Notebook luu
`TRAINING_SMOKE_TEST=False`; local validation override bang environment ma khong sua scientific default.

In [ ]:
import os
from pathlib import Path

SEED = 20260816
BATCH_SIZE = 256
MAX_EPOCHS = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 7
GRADIENT_CLIP_NORM = 1.0
HIDDEN_SIZE = 64
NUM_LAYERS = 1
NUM_WORKERS = 0

TRAINING_SMOKE_TEST = False
TRAINING_SMOKE_TEST = os.getenv(
    "GREENHOUSE_TRAINING_SMOKE_TEST", str(TRAINING_SMOKE_TEST)
).lower() in {"1", "true", "yes"}
SMOKE_MAX_EPOCHS = 1
SMOKE_MAX_TRAIN_BATCHES = 3
SMOKE_MAX_EVAL_BATCHES = 2

default_data_root = Path("/content/smart_greenhouse_dataset") if Path("/content").exists() else Path.cwd()
DATA_ROOT = Path(os.getenv("GREENHOUSE_DATA_ROOT", str(default_data_root))).expanduser()
PREPROCESSING_ARTIFACT_DIR = Path(
    os.getenv(
        "GREENHOUSE_PREPROCESSING_ARTIFACT_DIR",
        str(DATA_ROOT / "artifacts" / "preprocessing"),
    )
)
MODEL_ARTIFACT_DIR = Path(
    os.getenv(
        "GREENHOUSE_MODEL_ARTIFACT_DIR",
        str(DATA_ROOT / "artifacts" / (
            "model_training_smoke" if TRAINING_SMOKE_TEST else "model_training"
        )),
    )
)
INDEX_FILE = DATA_ROOT / "full_dataset_index.csv"

EXPECTED_SCENARIOS = 24
EXPECTED_ROWS_PER_SCENARIO = 70_128
EXPECTED_TOTAL_ROWS = 1_683_072
EXPECTED_START = "2018-01-01 00:00:00"
EXPECTED_END = "2025-12-31 23:00:00"
LOCKED_WINDOW_COUNTS = {
    "train": 1_051_200,
    "validation": 175_680,
    "temporal_test": 175_200,
    "scenario_test": 245_376,
    "combined_test": 35_040,
}
print(f"DATA_ROOT={DATA_ROOT.resolve()}")
print(f"TRAINING_SMOKE_TEST={TRAINING_SMOKE_TEST}")

## 02 - Imports

Chi dung cac package pho bien tren Colab. Notebook khong reinstall PyTorch/CUDA va khong can utility
package rieng tu workspace.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
import json
import math
import platform
import random
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, RandomSampler, SequentialSampler

## 03 - Reproducibility Setup

Seed Python, NumPy, PyTorch va DataLoader. Cac CUDA deterministic flags co chi phi cao khong bi ep bat;
policy nay uu tien reproducible initialization/sampling trong baseline research.

In [ ]:
def set_reproducibility(seed: int) -> torch.Generator:
    """Seed common random generators and return a seeded DataLoader generator."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator


data_loader_generator = set_reproducibility(SEED)

## 04 - Device / Colab Runtime Detection

Preprocessing arrays o CPU; model va batch chi duoc chuyen sang CUDA trong train/evaluation loop neu GPU
ton tai. Local smoke van hoat dong tren CPU.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 05 - Dataset and Artifact Path Configuration

Windows paths trong index duoc normalize cho Linux/Colab. Resolver khong glob directory; fallback chi
thu cac vi tri ro rang va log warning de tranh doc nham artifact V1.

In [ ]:
def normalize_index_path(raw_path: str) -> Path:
    normalized = str(raw_path).strip().replace("\\", "/")
    if not normalized:
        raise ValueError("Canonical index contains an empty path")
    return Path(normalized)


def resolve_scenario_path(raw_path: str, data_root: Path) -> Path:
    normalized = normalize_index_path(raw_path)
    candidates = [
        normalized if normalized.is_absolute() else data_root / normalized,
        data_root / "outputs" / "full_generation" / "ml" / normalized.name,
        data_root / "ml" / normalized.name,
        data_root / normalized.name,
    ]
    unique_candidates = list(dict.fromkeys(path.resolve() for path in candidates))
    existing = [path for path in unique_candidates if path.is_file()]
    if unique_candidates[0] in existing:
        return unique_candidates[0]
    if len(existing) != 1:
        raise FileNotFoundError(
            f"Cannot uniquely resolve {raw_path!r}; existing candidates={existing}"
        )
    warnings.warn(f"Explicit path fallback used for {raw_path!r}: {existing[0]}")
    return existing[0]

## 06 - Load Preprocessing Artifacts

Training notebook chi `load` feature scaler, target scaler, split manifest va preprocessing config.
Khong co scaler `fit`/`partial_fit`. Full mode tu choi smoke-fitted artifact.

In [ ]:
PREPROCESSING_FILES = {
    "feature_scaler": PREPROCESSING_ARTIFACT_DIR / "feature_scaler.pkl",
    "target_scaler": PREPROCESSING_ARTIFACT_DIR / "target_scaler.pkl",
    "split_manifest": PREPROCESSING_ARTIFACT_DIR / "split_manifest.json",
    "preprocessing_config": PREPROCESSING_ARTIFACT_DIR / "preprocessing_config.json",
}
missing_artifacts = [str(path) for path in PREPROCESSING_FILES.values() if not path.is_file()]
if missing_artifacts:
    raise FileNotFoundError(
        "Locked preprocessing artifacts are required. Run notebook 01 in full mode or mount the "
        f"artifact directory first. Missing: {missing_artifacts}"
    )

feature_scaler = joblib.load(PREPROCESSING_FILES["feature_scaler"])
target_scaler = joblib.load(PREPROCESSING_FILES["target_scaler"])
split_manifest = json.loads(PREPROCESSING_FILES["split_manifest"].read_text(encoding="utf-8"))
preprocessing_config = json.loads(
    PREPROCESSING_FILES["preprocessing_config"].read_text(encoding="utf-8")
)
if not TRAINING_SMOKE_TEST and preprocessing_config.get("smoke_test_execution"):
    raise ValueError("Full training refuses preprocessing artifacts fitted by smoke execution")
for scaler_name, scaler in (("feature", feature_scaler), ("target", target_scaler)):
    if not all(hasattr(scaler, attribute) for attribute in ("mean_", "scale_", "transform")):
        raise TypeError(f"Loaded {scaler_name} scaler is not a fitted StandardScaler-compatible object")
print(f"Loaded locked preprocessing artifacts from {PREPROCESSING_ARTIFACT_DIR.resolve()}")

## 07 - Validate Training Contract

Feature order, target order, lookback, horizon, split IDs va dates phai khop notebook 01. Test sets la
locked; notebook nay khong chon lai held-out scenarios.

In [ ]:
FEATURE_COLUMNS = [
    "air_temperature", "air_humidity", "soil_temperature", "soil_moisture",
    "light_lux", "pump_state", "fan_state", "grow_light_state",
]
TARGET_COLUMNS = [
    "air_temperature", "air_humidity", "soil_temperature", "soil_moisture", "light_lux"
]
CONTINUOUS_FEATURE_COLUMNS = TARGET_COLUMNS.copy()
BINARY_FEATURE_COLUMNS = ["pump_state", "fan_state", "grow_light_state"]
SOURCE_COLUMNS = ["timestamp", *FEATURE_COLUMNS]

if preprocessing_config["feature_columns"] != FEATURE_COLUMNS:
    raise ValueError("Feature order differs from locked preprocessing contract")
if preprocessing_config["target_columns"] != TARGET_COLUMNS:
    raise ValueError("Target order differs from locked preprocessing contract")
if preprocessing_config["binary_feature_columns"] != BINARY_FEATURE_COLUMNS:
    raise ValueError("Binary actuator policy differs from locked contract")
LOOKBACK_STEPS = int(split_manifest["lookback_steps"])
FORECAST_HORIZON = int(split_manifest["forecast_horizon"])
if (LOOKBACK_STEPS, FORECAST_HORIZON) != (24, 1):
    raise ValueError("This baseline requires locked lookback=24 and horizon=1")

development_scenario_ids = list(split_manifest["development_scenario_ids"])
held_out_scenario_ids = list(split_manifest["held_out_scenario_ids"])
if len(development_scenario_ids) != 20 or len(held_out_scenario_ids) != 4:
    raise ValueError("Locked split must contain 20 development and 4 held-out scenarios")
if set(development_scenario_ids).intersection(held_out_scenario_ids):
    raise ValueError("Development and held-out identities overlap")

train_start, train_end = map(pd.Timestamp, split_manifest["train_date_range"])
validation_start, validation_end = map(pd.Timestamp, split_manifest["validation_date_range"])
temporal_test_start, temporal_test_end = map(
    pd.Timestamp, split_manifest["temporal_test_date_range"]
)
expected_ranges = {
    "train": (pd.Timestamp("2018-01-01 00:00"), pd.Timestamp("2023-12-31 23:00")),
    "validation": (pd.Timestamp("2024-01-01 00:00"), pd.Timestamp("2024-12-31 23:00")),
    "temporal_test": (pd.Timestamp("2025-01-01 00:00"), pd.Timestamp("2025-12-31 23:00")),
}
if (train_start, train_end) != expected_ranges["train"]:
    raise ValueError("Locked TRAIN range changed")
if (validation_start, validation_end) != expected_ranges["validation"]:
    raise ValueError("Locked validation range changed")
if (temporal_test_start, temporal_test_end) != expected_ranges["temporal_test"]:
    raise ValueError("Locked temporal-test range changed")
print("Locked feature, target, split and window contracts PASS")

## 08 - Load Canonical Dataset Index

`full_dataset_index.csv` la membership authority. Hai naming families duoc doi xu nhu nhau; khong glob
thu muc va khong dung metadata/config lam feature.

In [ ]:
INDEX_REQUIRED_COLUMNS = {
    "parameter_set_id", "ml_file", "ml_rows", "ml_hash", "config_hash", "validation_status"
}


def load_canonical_index(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"Canonical index not found: {path}")
    index = pd.read_csv(path)
    missing = INDEX_REQUIRED_COLUMNS.difference(index.columns)
    if missing:
        raise ValueError(f"Canonical index missing columns: {sorted(missing)}")
    if len(index) != EXPECTED_SCENARIOS:
        raise ValueError(f"Expected 24 scenarios, found {len(index)}")
    if index["parameter_set_id"].duplicated().any() or index["config_hash"].duplicated().any():
        raise ValueError("Canonical identities/configs must be unique")
    if not (index["ml_rows"].astype(int) == EXPECTED_ROWS_PER_SCENARIO).all():
        raise ValueError("Canonical index row-count mismatch")
    if not (index["validation_status"] == "PASS").all():
        raise ValueError("Canonical index contains a non-PASS scenario")
    canonical_ids = set(index["parameter_set_id"])
    locked_ids = set(development_scenario_ids + held_out_scenario_ids)
    if canonical_ids != locked_ids:
        raise ValueError("Locked split identities do not partition the canonical index")
    return index.sort_values("parameter_set_id").reset_index(drop=True)


canonical_index = load_canonical_index(INDEX_FILE)
assert int(canonical_index["ml_rows"].astype(int).sum()) == EXPECTED_TOTAL_ROWS
print(f"Canonical membership: {len(canonical_index)} scenarios, {EXPECTED_TOTAL_ROWS:,} rows")

## 09 - Resolve Canonical Scenario Files

Moi locked identity resolve dung mot indexed file. Duplicate resolved path bi tu choi de giu quan he
one-scenario/one-trajectory.

In [ ]:
scenario_paths = {
    row.parameter_set_id: resolve_scenario_path(row.ml_file, DATA_ROOT)
    for row in canonical_index.itertuples(index=False)
}
if len(set(scenario_paths.values())) != EXPECTED_SCENARIOS:
    raise ValueError("Multiple canonical identities resolved to the same file")
print(f"Resolved {len(scenario_paths)} canonical scenario files")

## 10 - Reconstruct Scenario Frames / Efficient Arrays

Moi CSV duoc validate day du, scale theo timestep mot lan, roi cache thanh contiguous NumPy arrays.
Binary actuator van 0/1. Khong concatenate scenarios va khong materialize sliding windows.

In [ ]:
@dataclass(frozen=True)
class ScenarioArrays:
    timestamps: np.ndarray
    scaled_features: np.ndarray
    scaled_targets: np.ndarray
    raw_targets: np.ndarray


def validate_source_frame(frame: pd.DataFrame, scenario_id: str) -> pd.DataFrame:
    if list(frame.columns) != SOURCE_COLUMNS:
        raise ValueError(f"{scenario_id}: exact schema mismatch")
    if len(frame) != EXPECTED_ROWS_PER_SCENARIO:
        raise ValueError(f"{scenario_id}: expected 70,128 rows, found {len(frame)}")
    frame = frame.copy()
    frame["timestamp"] = pd.to_datetime(frame["timestamp"], errors="raise")
    if frame["timestamp"].duplicated().any() or not frame["timestamp"].is_monotonic_increasing:
        raise ValueError(f"{scenario_id}: duplicate/nonascending timestamps")
    if frame["timestamp"].iloc[0] != pd.Timestamp(EXPECTED_START):
        raise ValueError(f"{scenario_id}: unexpected start")
    if frame["timestamp"].iloc[-1] != pd.Timestamp(EXPECTED_END):
        raise ValueError(f"{scenario_id}: unexpected end")
    if not (frame["timestamp"].diff().dropna() == pd.Timedelta(hours=1)).all():
        raise ValueError(f"{scenario_id}: non-hourly timestamp gap")
    numeric = frame[FEATURE_COLUMNS].to_numpy(np.float64)
    if frame[FEATURE_COLUMNS].isna().any().any() or not np.isfinite(numeric).all():
        raise ValueError(f"{scenario_id}: NaN/Inf detected")
    for column in BINARY_FEATURE_COLUMNS:
        if not set(frame[column].unique()).issubset({0, 1}):
            raise ValueError(f"{scenario_id}: {column} is not binary")
    return frame


def frame_to_arrays(frame: pd.DataFrame) -> ScenarioArrays:
    continuous = frame[CONTINUOUS_FEATURE_COLUMNS].to_numpy(np.float64)
    binary = frame[BINARY_FEATURE_COLUMNS].to_numpy(np.float32)
    scaled_continuous = feature_scaler.transform(continuous).astype(np.float32)
    scaled_features = np.ascontiguousarray(np.column_stack([scaled_continuous, binary]))
    scaled_targets = np.ascontiguousarray(target_scaler.transform(continuous).astype(np.float32))
    return ScenarioArrays(
        timestamps=np.ascontiguousarray(frame["timestamp"].to_numpy(dtype="datetime64[ns]")),
        scaled_features=scaled_features,
        scaled_targets=scaled_targets,
        raw_targets=np.ascontiguousarray(continuous.astype(np.float32)),
    )


active_development_ids = development_scenario_ids[:1] if TRAINING_SMOKE_TEST else development_scenario_ids
active_held_out_ids = held_out_scenario_ids[:1] if TRAINING_SMOKE_TEST else held_out_scenario_ids
active_scenario_ids = active_development_ids + active_held_out_ids
scenario_arrays: dict[str, ScenarioArrays] = {}
for scenario_id in active_scenario_ids:
    source_frame = validate_source_frame(pd.read_csv(scenario_paths[scenario_id]), scenario_id)
    scenario_arrays[scenario_id] = frame_to_arrays(source_frame)
print(f"Cached timestep arrays for {len(scenario_arrays)} scenarios: {active_scenario_ids}")

## 11 - Reconstruct Locked Sequence Indices

Sequence index chi luu scenario code va target position. Target timestamp quyet dinh split; validation/test
target duoc phep dung historical context truoc boundary. Full counts phai khop contract da khoa.

In [ ]:
@dataclass(frozen=True)
class SequenceIndex:
    split_name: str
    scenario_ids: tuple[str, ...]
    scenario_codes: np.ndarray
    target_positions: np.ndarray
    target_start: pd.Timestamp
    target_end: pd.Timestamp

    def __len__(self) -> int:
        return int(len(self.target_positions))

    def resolve(self, item: int) -> tuple[str, int]:
        return self.scenario_ids[int(self.scenario_codes[item])], int(self.target_positions[item])


def valid_target_positions(
    arrays: ScenarioArrays,
    target_start: pd.Timestamp,
    target_end: pd.Timestamp,
) -> np.ndarray:
    timestamps = arrays.timestamps
    positions = np.flatnonzero(
        (timestamps >= np.datetime64(target_start)) & (timestamps <= np.datetime64(target_end))
    ).astype(np.int64)
    input_end = positions - FORECAST_HORIZON
    input_start = input_end - LOOKBACK_STEPS + 1
    feasible = input_start >= 0
    positions, input_end, input_start = positions[feasible], input_end[feasible], input_start[feasible]
    continuous = (
        (timestamps[input_end] - timestamps[input_start] == np.timedelta64(LOOKBACK_STEPS - 1, "h"))
        & (timestamps[positions] - timestamps[input_end] == np.timedelta64(FORECAST_HORIZON, "h"))
    )
    return positions[continuous].astype(np.int32)


def build_sequence_index(
    arrays_by_scenario: dict[str, ScenarioArrays],
    scenario_ids: list[str],
    split_name: str,
    target_start: pd.Timestamp,
    target_end: pd.Timestamp,
) -> SequenceIndex:
    ordered_ids = tuple(sorted(scenario_ids))
    code_chunks, position_chunks = [], []
    for scenario_code, scenario_id in enumerate(ordered_ids):
        positions = valid_target_positions(arrays_by_scenario[scenario_id], target_start, target_end)
        code_chunks.append(np.full(len(positions), scenario_code, dtype=np.int16))
        position_chunks.append(positions)
    return SequenceIndex(
        split_name=split_name,
        scenario_ids=ordered_ids,
        scenario_codes=np.concatenate(code_chunks),
        target_positions=np.concatenate(position_chunks),
        target_start=target_start,
        target_end=target_end,
    )


if TRAINING_SMOKE_TEST:
    smoke_ranges = {
        "train": (pd.Timestamp("2018-01-01 00:00"), pd.Timestamp("2018-01-14 23:00")),
        "validation": (pd.Timestamp("2024-01-01 00:00"), pd.Timestamp("2024-01-07 23:00")),
        "temporal_test": (pd.Timestamp("2025-01-01 00:00"), pd.Timestamp("2025-01-07 23:00")),
    }
    active_train_start, active_train_end = smoke_ranges["train"]
    active_validation_start, active_validation_end = smoke_ranges["validation"]
    active_test_start, active_test_end = smoke_ranges["temporal_test"]
else:
    active_train_start, active_train_end = train_start, train_end
    active_validation_start, active_validation_end = validation_start, validation_end
    active_test_start, active_test_end = temporal_test_start, temporal_test_end

scenario_test_end = active_train_end if TRAINING_SMOKE_TEST else active_validation_end

sequence_indices = {
    "train": build_sequence_index(scenario_arrays, active_development_ids, "train", active_train_start, active_train_end),
    "validation": build_sequence_index(scenario_arrays, active_development_ids, "validation", active_validation_start, active_validation_end),
    "temporal_test": build_sequence_index(scenario_arrays, active_development_ids, "temporal_test", active_test_start, active_test_end),
    "scenario_test": build_sequence_index(scenario_arrays, active_held_out_ids, "scenario_test", active_train_start, scenario_test_end),
    "combined_test": build_sequence_index(scenario_arrays, active_held_out_ids, "combined_test", active_test_start, active_test_end),
}
actual_window_counts = {name: len(index) for name, index in sequence_indices.items()}
if not TRAINING_SMOKE_TEST and actual_window_counts != LOCKED_WINDOW_COUNTS:
    raise ValueError(
        f"Locked window-count mismatch: expected={LOCKED_WINDOW_COUNTS}, actual={actual_window_counts}"
    )
if any(count == 0 for count in actual_window_counts.values()):
    raise ValueError("Every locked evaluation concept must contain windows")
print(f"Sequence window counts: {actual_window_counts}")

## 12 - Training DataLoader Construction

Optimized Dataset slice cached arrays khi `__getitem__`; no van lazy theo window. Training loader shuffle
windows sau split; validation/test dung sequential sampler. Numerical equivalence voi raw-scaler logic duoc assert.

In [ ]:
class OptimizedGreenhouseDataset(Dataset):
    def __init__(self, arrays: dict[str, ScenarioArrays], index: SequenceIndex) -> None:
        self.arrays = arrays
        self.index = index

    def __len__(self) -> int:
        return len(self.index)

    def __getitem__(self, item: int) -> tuple[torch.Tensor, torch.Tensor]:
        scenario_id, target_position = self.index.resolve(item)
        arrays = self.arrays[scenario_id]
        input_end = target_position - FORECAST_HORIZON
        input_start = input_end - LOOKBACK_STEPS + 1
        features = arrays.scaled_features[input_start : input_end + 1]
        target = arrays.scaled_targets[target_position]
        if features.shape != (LOOKBACK_STEPS, len(FEATURE_COLUMNS)):
            raise RuntimeError(f"Invalid sequence shape: {features.shape}")
        return torch.from_numpy(features), torch.from_numpy(target)


datasets = {
    name: OptimizedGreenhouseDataset(scenario_arrays, index)
    for name, index in sequence_indices.items()
}


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(dataset: Dataset, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
        worker_init_fn=seed_worker if NUM_WORKERS else None,
        generator=data_loader_generator if shuffle else None,
        drop_last=False,
    )


train_loader = make_loader(datasets["train"], shuffle=True)
validation_loader = make_loader(datasets["validation"], shuffle=False)
temporal_test_loader = make_loader(datasets["temporal_test"], shuffle=False)
scenario_test_loader = make_loader(datasets["scenario_test"], shuffle=False)
combined_test_loader = make_loader(datasets["combined_test"], shuffle=False)
loaders = {
    "train": train_loader,
    "validation": validation_loader,
    "temporal_test": temporal_test_loader,
    "scenario_test": scenario_test_loader,
    "combined_test": combined_test_loader,
}
assert isinstance(train_loader.sampler, RandomSampler)
assert all(isinstance(loaders[name].sampler, SequentialSampler) for name in loaders if name != "train")


def assert_optimized_equivalence(index: SequenceIndex, sample_items: list[int]) -> None:
    for item in sample_items:
        scenario_id, target_position = index.resolve(item)
        arrays = scenario_arrays[scenario_id]
        input_end = target_position - FORECAST_HORIZON
        input_start = input_end - LOOKBACK_STEPS + 1
        raw_sensor = arrays.raw_targets[input_start : input_end + 1].astype(np.float64)
        expected_sensor = feature_scaler.transform(raw_sensor).astype(np.float32)
        expected_binary = arrays.scaled_features[input_start : input_end + 1, 5:]
        expected_features = np.column_stack([expected_sensor, expected_binary]).astype(np.float32)
        expected_target = target_scaler.transform(
            arrays.raw_targets[[target_position]].astype(np.float64)
        ).astype(np.float32)[0]
        actual_features, actual_target = datasets[index.split_name][item]
        np.testing.assert_allclose(actual_features.numpy(), expected_features, rtol=1e-6, atol=1e-6)
        np.testing.assert_allclose(actual_target.numpy(), expected_target, rtol=1e-6, atol=1e-6)


assert_optimized_equivalence(sequence_indices["train"], [0, len(sequence_indices["train"]) // 2, -1])
sample_batch = next(iter(train_loader))
assert sample_batch[0].shape[1:] == (24, 8) and sample_batch[1].shape[1:] == (5,)
assert torch.isfinite(sample_batch[0]).all() and torch.isfinite(sample_batch[1]).all()
print(f"Optimized Dataset/DataLoader PASS: X={tuple(sample_batch[0].shape)}, Y={tuple(sample_batch[1].shape)}")

## 13 - Persistence Baseline

Persistence du bao sensor state t+1 bang state sensor cuoi cung tai t. No khong co parameter. Prediction
lay tu raw physical sensor state, khong nham feature-scaled values voi target-scaled values.

In [ ]:
def persistence_arrays(
    index: SequenceIndex,
    arrays_by_scenario: dict[str, ScenarioArrays],
    max_samples: int | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    sample_count = len(index) if max_samples is None else min(len(index), max_samples)
    predictions, targets = [], []
    for item in range(sample_count):
        scenario_id, target_position = index.resolve(item)
        arrays = arrays_by_scenario[scenario_id]
        input_end = target_position - FORECAST_HORIZON
        predictions.append(arrays.raw_targets[input_end])
        targets.append(arrays.raw_targets[target_position])
    return np.asarray(predictions, np.float32), np.asarray(targets, np.float32)


persistence_smoke_prediction, persistence_smoke_target = persistence_arrays(
    sequence_indices["validation"],
    scenario_arrays,
    max_samples=BATCH_SIZE,
)
assert persistence_smoke_prediction.shape == persistence_smoke_target.shape
assert persistence_smoke_prediction.shape[1] == len(TARGET_COLUMNS)
assert np.isfinite(persistence_smoke_prediction).all()
print(f"Persistence baseline shape PASS: {persistence_smoke_prediction.shape}")

## 14 - Model Architecture Definitions

GRU va LSTM cung input 8, hidden 64, 1 layer, output 5. Last hidden representation tom tat 24 gio va
duoc Linear head map thanh next-state prediction trong standardized target space.

In [ ]:
@dataclass(frozen=True)
class ModelConfig:
    input_size: int = 8
    hidden_size: int = 64
    num_layers: int = 1
    output_size: int = 5


class GRUForecaster(nn.Module):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        self.recurrent = nn.GRU(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            num_layers=config.num_layers,
            batch_first=True,
        )
        self.output_head = nn.Linear(config.hidden_size, config.output_size)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        _, final_hidden = self.recurrent(features)
        return self.output_head(final_hidden[-1])


class LSTMForecaster(nn.Module):
    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        self.recurrent = nn.LSTM(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            num_layers=config.num_layers,
            batch_first=True,
        )
        self.output_head = nn.Linear(config.hidden_size, config.output_size)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        _, (final_hidden, _) = self.recurrent(features)
        return self.output_head(final_hidden[-1])


model_config = ModelConfig()
assert model_config.input_size == len(FEATURE_COLUMNS)
assert model_config.output_size == len(TARGET_COLUMNS)

## 15 - Training Utilities

Mot generic loop duoc dung cho ca GRU/LSTM: AdamW, standardized MSE, finite checks, gradient clipping,
validation-only early stopping va atomic best-checkpoint save. Test loaders khong xuat hien trong `fit_model`.

In [ ]:
CHECKPOINT_DIR = MODEL_ARTIFACT_DIR / "checkpoints"
HISTORY_DIR = MODEL_ARTIFACT_DIR / "histories"
METRICS_DIR = MODEL_ARTIFACT_DIR / "metrics"
PLOT_DIR = MODEL_ARTIFACT_DIR / "plots"
for directory in (CHECKPOINT_DIR, HISTORY_DIR, METRICS_DIR, PLOT_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def assert_finite_gradients(model: nn.Module) -> None:
    for name, parameter in model.named_parameters():
        if parameter.grad is not None and not torch.isfinite(parameter.grad).all():
            raise FloatingPointError(f"Non-finite gradient detected in {name}")


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    max_batches: int | None = None,
) -> float:
    model.train()
    loss_sum, sample_count = 0.0, 0
    for batch_index, (features, targets) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        features = features.to(device, non_blocking=device.type == "cuda")
        targets = targets.to(device, non_blocking=device.type == "cuda")
        optimizer.zero_grad(set_to_none=True)
        predictions = model(features)
        if not torch.isfinite(predictions).all():
            raise FloatingPointError("Non-finite training prediction")
        loss = criterion(predictions, targets)
        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite training loss")
        loss.backward()
        assert_finite_gradients(model)
        nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
        optimizer.step()
        loss_sum += float(loss.detach()) * len(features)
        sample_count += len(features)
    if sample_count == 0:
        raise RuntimeError("Training loader produced zero samples")
    return loss_sum / sample_count


@torch.no_grad()
def evaluate_loss(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    max_batches: int | None = None,
) -> float:
    model.eval()
    loss_sum, sample_count = 0.0, 0
    for batch_index, (features, targets) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        features = features.to(device, non_blocking=device.type == "cuda")
        targets = targets.to(device, non_blocking=device.type == "cuda")
        predictions = model(features)
        loss = criterion(predictions, targets)
        if not torch.isfinite(predictions).all() or not torch.isfinite(loss):
            raise FloatingPointError("Non-finite validation prediction/loss")
        loss_sum += float(loss) * len(features)
        sample_count += len(features)
    if sample_count == 0:
        raise RuntimeError("Validation loader produced zero samples")
    return loss_sum / sample_count


def save_checkpoint_atomic(path: Path, payload: dict[str, object]) -> None:
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary_path)
    os.replace(temporary_path, path)


def load_torch_checkpoint(path: Path, map_location: torch.device | str) -> dict[str, object]:
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def fit_model(
    model_name: str,
    model: nn.Module,
    model_config: ModelConfig,
    training_loader: DataLoader,
    validation_loader: DataLoader,
    checkpoint_path: Path,
) -> dict[str, object]:
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    effective_epochs = SMOKE_MAX_EPOCHS if TRAINING_SMOKE_TEST else MAX_EPOCHS
    max_train_batches = SMOKE_MAX_TRAIN_BATCHES if TRAINING_SMOKE_TEST else None
    max_eval_batches = SMOKE_MAX_EVAL_BATCHES if TRAINING_SMOKE_TEST else None
    best_validation_loss = math.inf
    best_epoch = 0
    patience_counter = 0
    history: list[dict[str, float | int]] = []

    for epoch in range(1, effective_epochs + 1):
        epoch_start = time.perf_counter()
        train_loss = train_one_epoch(
            model, training_loader, optimizer, criterion, DEVICE, max_train_batches
        )
        validation_loss = evaluate_loss(
            model, validation_loader, criterion, DEVICE, max_eval_batches
        )
        duration = time.perf_counter() - epoch_start
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": validation_loss,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "epoch_duration_seconds": duration,
        })
        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_epoch = epoch
            patience_counter = 0
            save_checkpoint_atomic(checkpoint_path, {
                "model_name": model_name,
                "model_state_dict": model.state_dict(),
                "model_config": asdict(model_config),
                "epoch": epoch,
                "best_validation_loss": best_validation_loss,
                "feature_columns": FEATURE_COLUMNS,
                "target_columns": TARGET_COLUMNS,
                "lookback_steps": LOOKBACK_STEPS,
                "forecast_horizon": FORECAST_HORIZON,
                "seed": SEED,
            })
        else:
            patience_counter += 1
        print(
            f"{model_name} epoch {epoch:02d}: train={train_loss:.6f}, "
            f"validation={validation_loss:.6f}, seconds={duration:.2f}"
        )
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"{model_name}: early stopping at epoch {epoch}")
            break

    best_checkpoint = load_torch_checkpoint(checkpoint_path, DEVICE)
    if best_checkpoint["model_name"] != model_name:
        raise ValueError("Checkpoint architecture mismatch")
    model.load_state_dict(best_checkpoint["model_state_dict"])
    return {
        "model": model,
        "history": history,
        "best_epoch": best_epoch,
        "best_validation_loss": best_validation_loss,
        "checkpoint_path": checkpoint_path,
    }

## 16 - GRU Training

GRU baseline duoc train chi tren TRAIN windows. Local smoke chay mot epoch/ba batch de chung minh
forward, backward, finite gradient, optimizer step va checkpoint save.

In [ ]:
set_reproducibility(SEED)
data_loader_generator.manual_seed(SEED)
gru_model = GRUForecaster(model_config).to(DEVICE)
gru_result = fit_model(
    "GRU",
    gru_model,
    model_config,
    train_loader,
    validation_loader,
    CHECKPOINT_DIR / "best_gru.pt",
)
gru_model = gru_result["model"]

## 17 - GRU Validation

Best GRU checkpoint duoc danh gia lai tren validation 2024 (hoac smoke subset). Validation loss nay la
criterion hop le cho early stopping va model comparison; Test A/B/C van chua duoc mo.

In [ ]:
gru_validation_loss = evaluate_loss(
    gru_model,
    validation_loader,
    nn.MSELoss(),
    DEVICE,
    SMOKE_MAX_EVAL_BATCHES if TRAINING_SMOKE_TEST else None,
)
assert math.isfinite(gru_validation_loss)
print(f"Frozen GRU validation MSE: {gru_validation_loss:.6f}")

## 18 - LSTM Training

LSTM dung cung hidden size, layer count, optimizer, loss, batch va seed policy de so sanh cong bang voi
GRU. Num-layers=1 nen khong khai bao recurrent dropout gia.

In [ ]:
set_reproducibility(SEED)
data_loader_generator.manual_seed(SEED)
lstm_model = LSTMForecaster(model_config).to(DEVICE)
lstm_result = fit_model(
    "LSTM",
    lstm_model,
    model_config,
    train_loader,
    validation_loader,
    CHECKPOINT_DIR / "best_lstm.pt",
)
lstm_model = lstm_result["model"]

## 19 - LSTM Validation

Best LSTM checkpoint duoc danh gia tren cung validation protocol voi GRU. Khong test metric nao tham gia
buoc nay.

In [ ]:
lstm_validation_loss = evaluate_loss(
    lstm_model,
    validation_loader,
    nn.MSELoss(),
    DEVICE,
    SMOKE_MAX_EVAL_BATCHES if TRAINING_SMOKE_TEST else None,
)
assert math.isfinite(lstm_validation_loss)
print(f"Frozen LSTM validation MSE: {lstm_validation_loss:.6f}")

## 20 - Validation-based Model Comparison

Preferred baseline duoc chon duy nhat bang best validation loss. Sau dong nay architecture/epoch da
freeze; khong duoc doi quyet dinh sau khi xem Test A/B/C.

In [ ]:
validation_comparison = {
    "GRU": float(gru_result["best_validation_loss"]),
    "LSTM": float(lstm_result["best_validation_loss"]),
}
preferred_model_name = min(validation_comparison, key=validation_comparison.get)
preferred_model = gru_model if preferred_model_name == "GRU" else lstm_model
model_selection_record = {
    "criterion": "minimum validation standardized MSE",
    "validation_losses": validation_comparison,
    "preferred_model": preferred_model_name,
    "test_metrics_used_for_selection": False,
}
print(f"Preferred baseline frozen from validation only: {preferred_model_name}")

## 21 - Locked Final Evaluation

Chi sau model selection, frozen GRU/LSTM moi duoc evaluate tren Test A temporal, Test B scenario va Test C
combined. Persistence cung dung y het evaluation splits. Smoke gioi han vai batch, full run dung toan bo.

In [ ]:
@torch.no_grad()
def predict_scaled(
    model: nn.Module,
    loader: DataLoader,
    max_batches: int | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    predictions, targets = [], []
    for batch_index, (features, batch_targets) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        features = features.to(DEVICE, non_blocking=DEVICE.type == "cuda")
        batch_predictions = model(features).cpu().numpy()
        if not np.isfinite(batch_predictions).all():
            raise FloatingPointError("Non-finite final prediction")
        predictions.append(batch_predictions)
        targets.append(batch_targets.numpy())
    return np.concatenate(predictions), np.concatenate(targets)


def physical_metrics(
    true_physical: np.ndarray,
    predicted_physical: np.ndarray,
    true_scaled: np.ndarray,
    predicted_scaled: np.ndarray,
) -> dict[str, object]:
    if not all(np.isfinite(array).all() for array in (
        true_physical, predicted_physical, true_scaled, predicted_scaled
    )):
        raise FloatingPointError("Metrics received NaN/Inf")
    result: dict[str, object] = {
        "standardized_MSE": float(np.mean((true_scaled - predicted_scaled) ** 2)),
        "targets": {},
    }
    for column_index, column in enumerate(TARGET_COLUMNS):
        true_values = true_physical[:, column_index].astype(np.float64)
        predicted_values = predicted_physical[:, column_index].astype(np.float64)
        residual = true_values - predicted_values
        denominator = float(np.sum((true_values - true_values.mean()) ** 2))
        r2 = 0.0 if denominator <= np.finfo(np.float64).eps else 1.0 - float(np.sum(residual**2)) / denominator
        result["targets"][column] = {
            "MAE": float(np.mean(np.abs(residual))),
            "RMSE": float(np.sqrt(np.mean(residual**2))),
            "R2": float(r2),
        }
    return result


evaluation_loaders = {
    "validation": validation_loader,
    "temporal_test": temporal_test_loader,
    "scenario_test": scenario_test_loader,
    "combined_test": combined_test_loader,
}
evaluation_indices = {name: sequence_indices[name] for name in evaluation_loaders}
evaluation_models = {"GRU": gru_model, "LSTM": lstm_model}
max_eval_batches = SMOKE_MAX_EVAL_BATCHES if TRAINING_SMOKE_TEST else None
max_persistence_samples = BATCH_SIZE * SMOKE_MAX_EVAL_BATCHES if TRAINING_SMOKE_TEST else None
all_metrics: dict[str, dict[str, dict[str, object]]] = {"Persistence": {}, "GRU": {}, "LSTM": {}}
representative_predictions: dict[str, np.ndarray] = {}
representative_targets: np.ndarray | None = None

for split_name, index in evaluation_indices.items():
    persistence_prediction, persistence_target = persistence_arrays(
        index, scenario_arrays, max_persistence_samples
    )
    persistence_prediction_scaled = target_scaler.transform(
        persistence_prediction.astype(np.float64)
    ).astype(np.float32)
    persistence_target_scaled = target_scaler.transform(
        persistence_target.astype(np.float64)
    ).astype(np.float32)
    all_metrics["Persistence"][split_name] = physical_metrics(
        persistence_target,
        persistence_prediction,
        persistence_target_scaled,
        persistence_prediction_scaled,
    )

for model_name, model in evaluation_models.items():
    for split_name, loader in evaluation_loaders.items():
        predicted_scaled, true_scaled = predict_scaled(model, loader, max_eval_batches)
        predicted_physical = target_scaler.inverse_transform(predicted_scaled)
        true_physical = target_scaler.inverse_transform(true_scaled)
        all_metrics[model_name][split_name] = physical_metrics(
            true_physical, predicted_physical, true_scaled, predicted_scaled
        )
        if split_name == "validation":
            representative_predictions[model_name] = predicted_physical[:72]
            if representative_targets is None:
                representative_targets = true_physical[:72]
print("Locked final evaluation complete; test metrics were not used for selection")

## 22 - Physical-unit Metrics

Standardized MSE phuc vu comparison; MAE/RMSE/R2 duoc tinh rieng tung target sau inverse transform ve
don vi vat ly. Khong average truc tiep °C, %RH, VWC va lux; MAPE khong dung vi lux co the bang 0.

In [ ]:
def flatten_metric_row(model_name: str, split_name: str, metrics: dict[str, object]) -> dict[str, object]:
    row: dict[str, object] = {
        "Model": model_name,
        "Split": split_name,
        "standardized_MSE": metrics["standardized_MSE"],
    }
    for target_name, target_metrics in metrics["targets"].items():
        for metric_name, value in target_metrics.items():
            row[f"{target_name}_{metric_name}"] = value
    return row


model_comparison = pd.DataFrame([
    flatten_metric_row(model_name, split_name, split_metrics)
    for model_name, model_metrics in all_metrics.items()
    for split_name, split_metrics in model_metrics.items()
])
metric_values = model_comparison.select_dtypes(include=[np.number]).to_numpy()
if not np.isfinite(metric_values).all():
    raise FloatingPointError("Metric table contains NaN/Inf")
if TRAINING_SMOKE_TEST:
    print("Smoke metric functions PASS; values are not scientific training results")
else:
    print(model_comparison.to_string(index=False))

## 23 - Training Curves / Prediction Diagnostics

Bat buoc luu GRU/LSTM train-vs-validation loss. Prediction diagnostic dung deterministic first 72 validation
windows, khong cherry-pick theo performance, va hien thi physical units.

In [ ]:
def plot_loss_curve(history: list[dict[str, object]], model_name: str, output_path: Path) -> None:
    epochs = [record["epoch"] for record in history]
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, [record["train_loss"] for record in history], label="Train")
    plt.plot(epochs, [record["validation_loss"] for record in history], label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Standardized MSE")
    plt.title(f"{model_name} training history")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


plot_loss_curve(gru_result["history"], "GRU", PLOT_DIR / "gru_loss_curve.png")
plot_loss_curve(lstm_result["history"], "LSTM", PLOT_DIR / "lstm_loss_curve.png")

if representative_targets is None:
    raise RuntimeError("Representative validation targets were not captured")
representative_prediction = representative_predictions[preferred_model_name]
hours = np.arange(len(representative_prediction))
figure, axes = plt.subplots(len(TARGET_COLUMNS), 1, figsize=(11, 13), sharex=True)
for target_index, (axis, target_name) in enumerate(zip(axes, TARGET_COLUMNS)):
    axis.plot(hours, representative_targets[:, target_index], label="Actual", linewidth=1.5)
    axis.plot(hours, representative_prediction[:, target_index], label="Predicted", linewidth=1.2)
    axis.set_ylabel(target_name)
    axis.grid(alpha=0.2)
axes[0].legend()
axes[-1].set_xlabel("Deterministic validation-window offset (hours)")
figure.suptitle(f"{preferred_model_name}: first 72 validation predictions in physical units")
figure.tight_layout()
figure.savefig(PLOT_DIR / "preferred_model_validation_72h.png", dpi=150)
plt.close(figure)

## 24 - Save Model Artifacts

Save histories, model configs, physical metrics, comparison table va run manifest. Smoke artifacts vao
directory rieng, khong gia lam full-trained checkpoints.

In [ ]:
def write_json(path: Path, payload: object) -> None:
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


write_json(HISTORY_DIR / "gru_training_history.json", gru_result["history"])
write_json(HISTORY_DIR / "lstm_training_history.json", lstm_result["history"])
write_json(METRICS_DIR / "persistence_metrics.json", all_metrics["Persistence"])
write_json(METRICS_DIR / "gru_metrics.json", all_metrics["GRU"])
write_json(METRICS_DIR / "lstm_metrics.json", all_metrics["LSTM"])
model_comparison.to_csv(METRICS_DIR / "model_comparison.csv", index=False)
write_json(MODEL_ARTIFACT_DIR / "gru_model_config.json", asdict(model_config))
write_json(MODEL_ARTIFACT_DIR / "lstm_model_config.json", asdict(model_config))

training_run_manifest = {
    "seed": SEED,
    "device": str(DEVICE),
    "training_smoke_test": TRAINING_SMOKE_TEST,
    "model_config": asdict(model_config),
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "max_epochs": MAX_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "gradient_clip_norm": GRADIENT_CLIP_NORM,
    "feature_columns": FEATURE_COLUMNS,
    "target_columns": TARGET_COLUMNS,
    "preprocessing_artifact_dir": str(PREPROCESSING_ARTIFACT_DIR),
    "development_scenario_ids": development_scenario_ids,
    "held_out_scenario_ids": held_out_scenario_ids,
    "split_date_ranges": {
        "train": [str(train_start), str(train_end)],
        "validation": [str(validation_start), str(validation_end)],
        "temporal_test": [str(temporal_test_start), str(temporal_test_end)],
    },
    "locked_window_counts": LOCKED_WINDOW_COUNTS,
    "actual_execution_window_counts": actual_window_counts,
    "model_selection": model_selection_record,
    "gru_best_epoch": gru_result["best_epoch"],
    "gru_best_validation_loss": gru_result["best_validation_loss"],
    "lstm_best_epoch": lstm_result["best_epoch"],
    "lstm_best_validation_loss": lstm_result["best_validation_loss"],
}
write_json(MODEL_ARTIFACT_DIR / "training_run_manifest.json", training_run_manifest)
print(f"Training artifacts saved under {MODEL_ARTIFACT_DIR.resolve()}")

## 25 - Reload Checkpoint Verification

Moi best checkpoint duoc load vao model moi va chay cung deterministic validation batch. Prediction truoc
va sau reload phai allclose; neu khong, artifact khong duoc coi la reusable.

In [ ]:
deterministic_validation_batch = next(iter(validation_loader))[0].to(DEVICE)


@torch.no_grad()
def verify_checkpoint_reload(
    model_name: str,
    trained_model: nn.Module,
    checkpoint_path: Path,
) -> dict[str, object]:
    trained_model.eval()
    reference_prediction = trained_model(deterministic_validation_batch).cpu()
    checkpoint = load_torch_checkpoint(checkpoint_path, "cpu")
    checkpoint_config = ModelConfig(**checkpoint["model_config"])
    if model_name == "GRU":
        reloaded_model = GRUForecaster(checkpoint_config)
    elif model_name == "LSTM":
        reloaded_model = LSTMForecaster(checkpoint_config)
    else:
        raise ValueError(f"Unsupported model name: {model_name}")
    if checkpoint["model_name"] != model_name:
        raise ValueError("Checkpoint model identity mismatch")
    reloaded_model.load_state_dict(checkpoint["model_state_dict"])
    reloaded_model = reloaded_model.to(DEVICE).eval()
    reloaded_prediction = reloaded_model(deterministic_validation_batch).cpu()
    torch.testing.assert_close(reference_prediction, reloaded_prediction, rtol=1e-6, atol=1e-7)
    return {
        "status": "PASS",
        "checkpoint": str(checkpoint_path),
        "prediction_shape": list(reloaded_prediction.shape),
    }


checkpoint_reload_audit = {
    "GRU": verify_checkpoint_reload("GRU", gru_model, CHECKPOINT_DIR / "best_gru.pt"),
    "LSTM": verify_checkpoint_reload("LSTM", lstm_model, CHECKPOINT_DIR / "best_lstm.pt"),
}
print(f"Checkpoint reload verification PASS: {checkpoint_reload_audit}")

## 26 - Final Experiment Summary

Structured summary la machine-checkable gate. Local smoke khong duoc dien giai thanh accuracy result;
full Colab run moi tao scientific metrics va trained baseline artifacts chinh thuc.

In [ ]:
experiment_summary = {
    "status": "PASS",
    "canonical_scenarios": len(canonical_index),
    "canonical_rows": EXPECTED_TOTAL_ROWS,
    "development_scenarios": len(development_scenario_ids),
    "held_out_scenarios": len(held_out_scenario_ids),
    "features": len(FEATURE_COLUMNS),
    "targets": len(TARGET_COLUMNS),
    "lookback_steps": LOOKBACK_STEPS,
    "forecast_horizon": FORECAST_HORIZON,
    "locked_window_counts": LOCKED_WINDOW_COUNTS,
    "actual_execution_window_counts": actual_window_counts,
    "scaler_refit_performed": False,
    "optimized_arrays": True,
    "lazy_windows": True,
    "persistence_status": "PASS",
    "gru_forward_backward": "PASS",
    "lstm_forward_backward": "PASS",
    "checkpoint_reload": checkpoint_reload_audit,
    "model_selection": model_selection_record,
    "training_smoke_test": TRAINING_SMOKE_TEST,
    "full_gpu_training_executed": not TRAINING_SMOKE_TEST,
    "artifact_dir": str(MODEL_ARTIFACT_DIR),
}
print(json.dumps(experiment_summary, indent=2))